# CiviConnect Photo Assistant: train on your own photos

This notebook teaches the Photo Assistant your city's problems. It runs on **Google Colab** (free).

**How it works**
1. Every photo goes through **CLIP** (the same model the app uses) and becomes a list of 512 numbers, its *embedding*.
2. A small **logistic regression** layer learns which embeddings belong to which category. This is called a *linear probe*.
3. We compare it with the **zero-shot** version the app uses today (no training, only text descriptions) on photos the model never saw.
4. We export two small files, `civic_head.npz` and `metrics.json`. Put them in `ml/models/`, restart the service, and the app switches to **trained** mode automatically.

**Your dataset** goes in Google Drive, one folder per category (folder names must match exactly):

```
MyDrive/civic_dataset/
    pothole/        100-300 photos
    drainage/
    garbage/
    fallen_tree/
    streetlight/
    other/          (optional) broken footpaths, signboards, leaks...
    not_civic/      selfies, food, rooms, screenshots, pets...
```

Run the cells from top to bottom with **Runtime → Run all**. Turn on a GPU first (**Runtime → Change runtime type → T4 GPU**) to make step 4 faster.

## 1. Install and connect Google Drive

In [ ]:
!pip -q install "transformers>=4.44,<5" scikit-learn
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/civic_dataset"
OUT_DIR = "/content/drive/MyDrive/civic_model"

## 2. Settings

In [ ]:
import os, json, time, random
from pathlib import Path
import numpy as np

CLIP_ID = "openai/clip-vit-base-patch32"   # must match the app (ml/app/model.py)
TEST_SHARE = 0.2                            # 20% of photos are kept aside for the final exam
SEED = 42
CLASSES_KNOWN = ["pothole", "drainage", "garbage", "fallen_tree", "streetlight", "other", "not_civic"]
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# The zero-shot prompts. Keep in sync with ml/app/labels.py.
PROMPTS = {
    "pothole": ["a photo of a pothole in the road", "a damaged road with a deep pothole", "a broken asphalt street with holes and cracks", "a muddy pothole filled with rain water on an Indian road"],
    "drainage": ["a photo of a blocked drain overflowing onto the street", "an open sewer drain with dirty water", "a flooded street because of a clogged drainage", "a broken manhole with sewage water"],
    "garbage": ["a photo of a garbage dump on the roadside", "a pile of trash and plastic waste on the street", "an overflowing garbage bin", "uncollected waste dumped in an empty plot"],
    "fallen_tree": ["a photo of a fallen tree blocking the road", "a big tree branch broken and lying on the street", "an uprooted tree after a storm", "a tree that fell on a footpath"],
    "streetlight": ["a photo of a street light pole", "a broken street lamp on a road", "a dark street at night with a street light not working", "a damaged lamp post with hanging wires"],
    "other": ["a photo of a broken footpath or sidewalk", "a damaged public signboard", "a leaking water pipe on the street", "stagnant water and mosquitoes in a public place", "broken public property in a city"],
    "not_civic": ["a selfie of a person", "a photo of food on a plate", "a screenshot of a phone screen", "a photo of a cat or a dog", "a photo of a room inside a house", "a photo of a document or text", "a blank or black image", "a photo of a car in a showroom"],
}
random.seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Look at the dataset

In [ ]:
from PIL import Image, ImageOps

files, targets = [], []
found = {}
for cls in sorted(os.listdir(DATA_DIR)):
    folder = Path(DATA_DIR) / cls
    if not folder.is_dir():
        continue
    if cls not in CLASSES_KNOWN:
        print(f"Skipping folder '{cls}': not one of {CLASSES_KNOWN}")
        continue
    # Photos can sit in sub-folders too (Kaggle downloads often have e.g. pothole/images/...).
    photos = sorted(f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in EXTS)
    found[cls] = len(photos)
    for f in photos:
        files.append(str(f)); targets.append(cls)

print("Photos found in each folder:")
for c in CLASSES_KNOWN:
    print(f"  {c:12s} {found.get(c, 'folder missing')}")
CLASSES = [c for c in CLASSES_KNOWN if c in set(targets)]
counts = {c: targets.count(c) for c in CLASSES}
print("Total photos:", len(files))
assert len(CLASSES) >= 2, (
    "Need photos in at least two category folders. Put .jpg/.png/.webp photos inside "
    f"{DATA_DIR}/pothole, {DATA_DIR}/garbage, ... and run again (Runtime > Run all)."
)
for c, n in counts.items():
    if n < 20:
        print(f"Warning: only {n} photos of '{c}'. Aim for 100+ for reliable results.")

def load(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(12, 2.4 * len(CLASSES)))
axes = np.atleast_2d(axes)
for r, c in enumerate(CLASSES):
    picks = [f for f, t in zip(files, targets) if t == c][:5]
    for k in range(5):
        ax = axes[r, k]; ax.axis("off")
        if k < len(picks):
            try:
                ax.imshow(load(picks[k]))
            except Exception:
                ax.text(0.5, 0.5, "unreadable file", ha="center", va="center", fontsize=8)
            ax.set_title(c if k == 0 else "", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Turn every photo into a CLIP embedding
This is the slow step: about 1 minute per 1,000 photos on a T4 GPU.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

def features(out):
    if hasattr(out, "shape"):
        return out
    for key in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(out, key, None) is not None:
            return getattr(out, key)
    raise TypeError("Unexpected CLIP output")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
processor = CLIPProcessor.from_pretrained(CLIP_ID)
print("Running on", device)

def normalize(v):
    return v / np.clip(np.linalg.norm(v, axis=-1, keepdims=True), 1e-8, None)

@torch.inference_mode()
def embed_images(paths, batch=32):
    out, bad = [], []
    for i in range(0, len(paths), batch):
        imgs, ok = [], []
        for p in paths[i:i + batch]:
            try:
                imgs.append(load(p)); ok.append(True)
            except Exception:
                imgs.append(Image.new("RGB", (224, 224))); ok.append(False); bad.append(p)
        inputs = processor(images=imgs, return_tensors="pt").to(device)
        out.append(features(model.get_image_features(**inputs)).float().cpu().numpy())
        print(f"\r{min(i + batch, len(paths))}/{len(paths)} photos", end="")
    print()
    if bad:
        print("Could not read (excluded):", bad)
    return normalize(np.concatenate(out)), set(bad)

@torch.inference_mode()
def embed_texts(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    return normalize(features(model.get_text_features(**inputs)).float().cpu().numpy())

started = time.time()
X, bad = embed_images(files)
keep = [i for i, f in enumerate(files) if f not in bad]
X = X[keep]; y = np.array([targets[i] for i in keep]); kept_files = [files[i] for i in keep]
print(f"Embeddings: {X.shape} in {time.time() - started:.0f} s")

## 5. Split into training and test photos
The test photos are never shown to the model during training, so the scores are honest.

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=TEST_SHARE, random_state=SEED, stratify=y)
X_tr, X_te, y_tr, y_te = X[tr], X[te], y[tr], y[te]
print("train:", len(tr), "| test:", len(te))

## 6. Baseline: zero-shot (what the app does today, no training)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

T = np.stack([normalize(embed_texts(PROMPTS[c]).mean(axis=0)) for c in CLASSES])
scale = float(model.logit_scale.exp().item())
zs_pred = np.array(CLASSES)[np.argmax(scale * X_te @ T.T, axis=1)]
zs_acc = accuracy_score(y_te, zs_pred)
zs_f1 = f1_score(y_te, zs_pred, average="macro")
print(f"Zero-shot accuracy: {zs_acc:.1%}   macro F1: {zs_f1:.3f}")

## 7. Train the linear probe
A short search picks how strongly to regularise (the `C` value) using 5-fold cross-validation on the training photos only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

folds = max(2, min(5, int(np.bincount(np.unique(y_tr, return_inverse=True)[1]).min())))
search = GridSearchCV(
    LogisticRegression(max_iter=3000, class_weight="balanced"),
    {"C": [0.5, 1, 2, 5, 10, 20]},
    cv=StratifiedKFold(folds, shuffle=True, random_state=SEED),
    scoring="f1_macro",
)
search.fit(X_tr, y_tr)
clf = search.best_estimator_
print("Best C:", search.best_params_["C"], "| cross-validated macro F1:", round(search.best_score_, 3))

pred = clf.predict(X_te)
acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred, average="macro")
print(f"Trained accuracy: {acc:.1%}   macro F1: {f1:.3f}   (zero-shot was {zs_acc:.1%})")
print(classification_report(y_te, pred, labels=list(clf.classes_), digits=3, zero_division=0))

## 8. Confusion matrix
Rows are the true category, columns what the model said. Everything off the diagonal is a mistake.

In [ ]:
cm = confusion_matrix(y_te, pred, labels=list(clf.classes_))
fig, ax = plt.subplots(figsize=(1.1 * len(clf.classes_) + 2, 1.1 * len(clf.classes_) + 1.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(clf.classes_)), clf.classes_, rotation=40, ha="right")
ax.set_yticks(range(len(clf.classes_)), clf.classes_)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Linear probe on {len(y_te)} test photos")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=160); plt.show()

## 9. The photos it got wrong
Useful to spot mislabelled photos in the dataset.

In [ ]:
wrong = [k for k in range(len(te)) if pred[k] != y_te[k]][:12]
if wrong:
    cols = 4; rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows)); axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, k in zip(axes, wrong):
        ax.imshow(load(kept_files[te[k]])); ax.set_title(f"true {y_te[k]}\nsaid {pred[k]}", fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("No mistakes on the test photos.")

## 10. Export for the app
Training is finished; the final model is refit on **all** photos so nothing is wasted. The scores above, from the held-out test photos, are the ones to report.

In [ ]:
final = LogisticRegression(max_iter=3000, class_weight="balanced", C=search.best_params_["C"]).fit(X, y)
classes = [str(c) for c in final.classes_]
W, b = final.coef_.astype(np.float32), final.intercept_.astype(np.float32)
if len(classes) == 2:  # binary models store one row; expand to one row per class for the app
    W = np.vstack([np.zeros_like(W), W]); b = np.array([0.0, b[0]], dtype=np.float32)
np.savez(f"{OUT_DIR}/civic_head.npz", W=W, b=b, classes=np.array(classes))

report = classification_report(y_te, pred, labels=classes, output_dict=True, zero_division=0)
metrics = {
    "encoder": CLIP_ID,
    "method": "CLIP embeddings + logistic regression (linear probe)",
    "classes": classes,
    "accuracy": round(float(acc), 4),
    "macroF1": round(float(f1), 4),
    "zeroShotAccuracy": round(float(zs_acc), 4),
    "zeroShotMacroF1": round(float(zs_f1), 4),
    "perClass": {c: {"precision": round(report[c]["precision"], 4), "recall": round(report[c]["recall"], 4), "f1": round(report[c]["f1-score"], 4), "support": int(report[c]["support"])} for c in classes},
    "confusion": {"labels": classes, "matrix": confusion_matrix(y_te, pred, labels=classes).tolist()},
    "nPhotos": int(len(y)), "nTrain": int(len(tr)), "nTest": int(len(te)),
    "regularisationC": search.best_params_["C"],
    "trainedAt": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(f"{OUT_DIR}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps({k: metrics[k] for k in ["accuracy", "macroF1", "zeroShotAccuracy", "nPhotos"]}, indent=2))
print("Saved to", OUT_DIR)

## 11. Download the two files

In [ ]:
from google.colab import files as colab_files
colab_files.download(f"{OUT_DIR}/civic_head.npz")
colab_files.download(f"{OUT_DIR}/metrics.json")

## 12. Put them in the app
1. Copy `civic_head.npz` and `metrics.json` into the project's `ml/models/` folder.
2. Restart the Photo Assistant (stop it with Ctrl+C and run `uvicorn app.main:app --port 8000` again).
3. It prints `trained mode`. The Admin page's Model card now shows these scores.

To go back to zero-shot, delete the two files and restart.